In [38]:
import networkx as nx
from pyvis.network import Network
import os

def visualize_interactive_with_legend(file_path, output_file="graph_legend.html"):
    """
    Generates an interactive PyVis graph and injects a custom HTML Legend 
    to explain what each color means.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return

    # 1. Load the Graph
    print(f"Loading {file_path}...")
    try:
        G = nx.read_graphml(file_path)
    except Exception as e:
        print(f"Error parsing GraphML: {e}")
        return

    # 2. Define a Color Palette manually
    # We map colors to entity types so we can build a consistent legend
    # You can add more colors to this list if you have many entity types
    hex_colors = [
        "#FF6666", # Red (e.g., Person)
        "#66FF66", # Green (e.g., Organization)
        "#66CCFF", # Blue (e.g., Technology)
        "#FFFF66", # Yellow (e.g., Game)
        "#FFCC66", # Orange
        "#CC99FF", # Purple
        "#FF99CC", # Pink
        "#99CC99"  # Sage
    ]
    
    # Find unique entity types
    entity_types = set()
    for _, data in G.nodes(data=True):
        raw_type = data.get('entity_type', 'Unknown')
        clean_type = str(raw_type).replace('"', '')
        entity_types.add(clean_type)
    
    # Map each type to a color
    type_to_color = {}
    sorted_types = sorted(list(entity_types))
    for i, etype in enumerate(sorted_types):
        # Cycle through colors if we have more types than colors
        color = hex_colors[i % len(hex_colors)]
        type_to_color[etype] = color

    # 3. Initialize PyVis
    net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", select_menu=False)

    # 4. Add Nodes manually to apply our Specific Colors
    for node, data in G.nodes(data=True):
        # Get Type
        raw_type = data.get('entity_type', 'Unknown')
        clean_type = str(raw_type).replace('"', '')
        
        # Get Color
        assigned_color = type_to_color.get(clean_type, "#cccccc")
        
        # Clean Label
        label_text = str(node).replace('"', '')
        
        # Add to PyVis
        # We use title=... to show description on hover
        tooltip = data.get('description', clean_type)
        net.add_node(node, label=label_text, title=tooltip, color=assigned_color)

    # 5. Add Edges (if any existed, though your file has none)
    for source, target in G.edges():
        net.add_edge(source, target)


    net.force_atlas_2based(
        gravity=-50,
        central_gravity=0.01,
        spring_length=100,
        spring_strength=0.08,
        damping=0.9,
        overlap=0
    )


    # 6. Enable Physics
    net.toggle_physics(True)
    # net.show_buttons(filter_=['physics'])

    # 7. Save the initial HTML
    net.save_graph(output_file)


    # ============================================================
    # 8. THE MAGIC TRICK: Inject HTML Legend into the generated file
    # ============================================================
    
    # Create the HTML string for the legend
    legend_items = ""
    for etype, color in type_to_color.items():
        legend_items += f"""
        <div style="display: flex; align-items: center; margin-bottom: 5px;">
            <div style="width: 15px; height: 15px; background-color: {color}; margin-right: 8px; border: 1px solid #fff;"></div>
            <span style="color: white; font-family: sans-serif; font-size: 14px;">{etype}</span>
        </div>
        """

    legend_html = f"""
    <div id="custom-legend" style="
        position: absolute;
        top: 10px;
        right: 10px;
        background-color: rgba(0, 0, 0, 0.7);
        padding: 15px;
        border-radius: 8px;
        border: 1px solid #555;
        z-index: 1000;
        min-width: 150px;
    ">
        <h3 style="color: white; margin-top: 0; font-family: sans-serif; border-bottom: 1px solid #555; padding-bottom: 5px;">Entity Legend</h3>
        {legend_items}
    </div>
    """

    # Read the generated file
    with open(output_file, "r", encoding='utf-8') as f:
        content = f.read()

    # Insert the legend right before the closing body tag
    new_content = content.replace("</body>", f"{legend_html}</body>")

    # Write it back
    with open(output_file, "w", encoding='utf-8') as f:
        f.write(new_content)

    print(f"Success! Interactive graph with Legend saved to: {os.path.abspath(output_file)}")

# --- USAGE ---
path = "/home/gatv-projects/Desktop/project/knowledge_sanitization/cache/big_merge_result.graphml" # Replace with your path
visualize_interactive_with_legend(path)

Loading /home/gatv-projects/Desktop/project/knowledge_sanitization/cache/big_merge_result.graphml...
Success! Interactive graph with Legend saved to: /home/gatv-projects/Desktop/project/playground/knowledge_graph_build/graph_legend.html


In [23]:
import os
import html
import math
import networkx as nx
from pyvis.network import Network


SEP = "<SEP>"


def _clean_text(value, max_len=None, multiline=False):
    """Clean GraphML text for labels/tooltips."""
    if value is None:
        return ""

    text = html.unescape(str(value)).strip()

    # Remove wrapping quotes repeatedly
    while len(text) >= 2 and text[0] == text[-1] and text[0] in {'"', "'", "“", "”", "‘", "’"}:
        text = text[1:-1].strip()

    if multiline:
        parts = [p.strip() for p in text.split(SEP) if p.strip()]
        text = "<br>".join(parts)
    else:
        parts = [p.strip() for p in text.split(SEP) if p.strip()]
        if parts:
            text = " | ".join(parts)

    if max_len and len(text) > max_len:
        text = text[: max_len - 3] + "..."

    return text


def _safe_entity_type(value):
    text = _clean_text(value)
    return text if text else "UNKNOWN"


def _build_type_color_map(entity_types):
    """
    Stable colors for common types, fallback palette for anything else.
    """
    fixed = {
        "PERSON": "#ff6b6b",
        "ORGANIZATION": "#4ecdc4",
        "GEO": "#54a0ff",
        "EVENT": "#feca57",
        "TECHNOLOGY": "#5f27cd",
        "PRODUCT": "#1dd1a1",
        "UNKNOWN": "#a5b1c2",
    }

    fallback = [
        "#ff9ff3",
        "#48dbfb",
        "#00d2d3",
        "#ff9f43",
        "#10ac84",
        "#ee5253",
        "#2e86de",
        "#8395a7",
    ]

    type_to_color = {}
    used_fallback = 0

    for etype in sorted(entity_types):
        if etype in fixed:
            type_to_color[etype] = fixed[etype]
        else:
            type_to_color[etype] = fallback[used_fallback % len(fallback)]
            used_fallback += 1

    return type_to_color


def _make_node_tooltip(node_id, data, degree):
    entity_type = _safe_entity_type(data.get("entity_type"))
    description = _clean_text(data.get("description"), max_len=2000, multiline=True)
    source_id = _clean_text(data.get("source_id"), max_len=800, multiline=True)

    tooltip = f"""
    <div style="max-width: 420px; white-space: normal;">
        <b>ID:</b> {html.escape(_clean_text(node_id, max_len=200))}<br>
        <b>Type:</b> {html.escape(entity_type)}<br>
        <b>Degree:</b> {degree}<br>
    """

    if description:
        tooltip += f"<b>Description:</b><br>{description}<br>"

    if source_id:
        tooltip += f"<b>Source ID:</b><br>{source_id}<br>"

    tooltip += "</div>"
    return tooltip


def _make_edge_tooltip(data):
    description = _clean_text(data.get("description"), max_len=1500, multiline=True)
    source_id = _clean_text(data.get("source_id"), max_len=600, multiline=True)
    weight = data.get("weight", "")

    tooltip = '<div style="max-width: 420px; white-space: normal;">'
    if description:
        tooltip += f"<b>Relation:</b><br>{description}<br>"
    if weight != "":
        tooltip += f"<b>Weight:</b> {html.escape(str(weight))}<br>"
    if source_id:
        tooltip += f"<b>Source ID:</b><br>{source_id}<br>"
    tooltip += "</div>"

    return tooltip


def _inject_legend(output_file, type_to_color):
    legend_items = ""
    for etype, color in sorted(type_to_color.items()):
        legend_items += f"""
        <div style="display:flex; align-items:center; margin-bottom:6px;">
            <div style="
                width:14px;
                height:14px;
                background-color:{color};
                margin-right:8px;
                border:1px solid #fff;
                border-radius:3px;
                flex-shrink:0;
            "></div>
            <span style="color:white; font-family:sans-serif; font-size:13px;">
                {html.escape(etype)}
            </span>
        </div>
        """

    legend_html = f"""
    <div id="custom-legend" style="
        position:absolute;
        top:10px;
        right:10px;
        background:rgba(0,0,0,0.78);
        padding:14px;
        border-radius:10px;
        border:1px solid #555;
        z-index:1000;
        min-width:170px;
        max-width:240px;
        box-shadow:0 4px 14px rgba(0,0,0,0.35);
    ">
        <h3 style="
            color:white;
            margin:0 0 10px 0;
            font-family:sans-serif;
            font-size:16px;
            border-bottom:1px solid #555;
            padding-bottom:6px;
        ">Entity Legend</h3>
        {legend_items}
    </div>
    """

    with open(output_file, "r", encoding="utf-8") as f:
        content = f.read()

    if "</body>" in content:
        content = content.replace("</body>", f"{legend_html}</body>")
    else:
        content += legend_html

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(content)


def visualize_interactive_with_legend(
    file_path,
    output_file="graph_legend.html",
    height="900px",
    width="100%",
    show_edge_labels=False,
    enable_physics_controls=False,
):
    """
    Visualize a GraphML knowledge graph with:
    - colors by entity_type
    - legend
    - degree-based node sizing
    - rich node/edge tooltips
    """

    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return

    print(f"Loading {file_path}...")
    try:
        G = nx.read_graphml(file_path)
    except Exception as e:
        print(f"Error parsing GraphML: {e}")
        return

    print(f"Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

    # Collect entity types
    entity_types = set()
    for _, data in G.nodes(data=True):
        entity_types.add(_safe_entity_type(data.get("entity_type")))

    type_to_color = _build_type_color_map(entity_types)

    net = Network(
        height=height,
        width=width,
        bgcolor="#1e1e1e",
        font_color="white",
        directed=G.is_directed(),
        notebook=False,
        select_menu=False,
        filter_menu=False,
    )

    # Better interaction defaults
    net.set_options("""
    const options = {
      "nodes": {
        "borderWidth": 1,
        "borderWidthSelected": 2,
        "shape": "dot",
        "font": {
          "size": 16,
          "face": "arial"
        },
        "scaling": {
          "min": 10,
          "max": 35
        }
      },
      "edges": {
        "color": {
          "inherit": false
        },
        "smooth": {
          "enabled": true,
          "type": "dynamic"
        },
        "arrows": {
          "to": {
            "enabled": false
          }
        },
        "font": {
          "size": 11,
          "align": "middle"
        }
      },
      "interaction": {
        "hover": true,
        "navigationButtons": true,
        "keyboard": true
      },
      "physics": {
        "enabled": true,
        "forceAtlas2Based": {
          "gravitationalConstant": -60,
          "centralGravity": 0.015,
          "springLength": 120,
          "springConstant": 0.05,
          "damping": 0.9,
          "avoidOverlap": 0.8
        },
        "solver": "forceAtlas2Based",
        "stabilization": {
          "enabled": true,
          "iterations": 300
        }
      }
    }
    """)

    degrees = dict(G.degree())

    # Add nodes
    for node, data in G.nodes(data=True):
        clean_type = _safe_entity_type(data.get("entity_type"))
        color = type_to_color.get(clean_type, "#a5b1c2")
        label = _clean_text(node, max_len=40)
        degree = degrees.get(node, 0)

        # Conservative size growth
        size = 12 + min(28, 4 * math.log2(degree + 1))

        tooltip = _make_node_tooltip(node, data, degree)

        net.add_node(
            node,
            label=label,
            title=tooltip,
            color=color,
            size=size,
        )

    # Add edges
    if G.is_multigraph():
        edge_iter = G.edges(keys=True, data=True)
        for source, target, _, data in edge_iter:
            edge_title = _make_edge_tooltip(data)
            edge_label = _clean_text(data.get("description"), max_len=40) if show_edge_labels else ""
            weight = data.get("weight", 1.0)
            try:
                value = float(weight)
            except Exception:
                value = 1.0

            net.add_edge(
                source,
                target,
                title=edge_title,
                label=edge_label,
                value=value,
                color="#848484",
            )
    else:
        for source, target, data in G.edges(data=True):
            edge_title = _make_edge_tooltip(data)
            edge_label = _clean_text(data.get("description"), max_len=40) if show_edge_labels else ""
            weight = data.get("weight", 1.0)
            try:
                value = float(weight)
            except Exception:
                value = 1.0

            net.add_edge(
                source,
                target,
                title=edge_title,
                label=edge_label,
                value=value,
                color="#848484",
            )

    if enable_physics_controls:
        net.show_buttons(filter_=["physics"])

    net.save_graph(output_file)
    _inject_legend(output_file, type_to_color)

    print(f"Success! Interactive graph saved to: {os.path.abspath(output_file)}")


# --- USAGE ---
path = "/home/gatv-projects/Desktop/project/knowledge_sanitization/cache/sanitized_global/graph_AetherNexus.graphml"
visualize_interactive_with_legend(
    path,
    output_file="graph_legend.html",
    show_edge_labels=False,
    enable_physics_controls=False,
)

Loading /home/gatv-projects/Desktop/project/knowledge_sanitization/cache/sanitized_global/graph_AetherNexus.graphml...
Loaded graph with 682 nodes and 1045 edges
Success! Interactive graph saved to: /home/gatv-projects/Desktop/project/playground/knowledge_graph_build/graph_legend.html
